In [1]:
# ============================================================
# CELL 0 — Install Dependencies
# ============================================================
!pip install stable-baselines3[extra] --quiet
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git --quiet
!pip install websockets>=15.0.0 --quiet


zsh:1: no matches found: stable-baselines3[extra]
zsh:1: 15.0.0 not found


In [14]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch as th
import torch.nn as nn
import math, json, os, sys, gymnasium, pickle
sys.path.append('.')

from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.monitor import Monitor
from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from gymnasium import spaces
from finrl.config import INDICATORS
from utils import (
    prepare_df, compute_metrics, compute_rolling_metrics,
    plot_metrics, compute_buy_and_hold, overfitting_check,
    check_degenerate_policy, check_lookahead_bias, regime_analysis
)

if th.backends.mps.is_available():
    DEVICE = th.device('mps')
elif th.cuda.is_available():
    DEVICE = th.device('cuda')
else:
    DEVICE = th.device('cpu')
print(f"Device: {DEVICE}")

SEEDS           = [1, 2, 3]
INITIAL_CAPITAL = 100_000
os.makedirs('./overlay_data/', exist_ok=True)
print("Imports complete ✅")


Device: mps
Imports complete ✅


In [4]:
# ============================================================
# CELL 2 — Architecture Definitions
# ============================================================

TICKERS = [
    'NVDA', 'AAPL', 'GOOGL', 'GOOG', 'MSFT', 'AMZN', 'META', 'AVGO', 'TSLA',
    'WMT', 'LLY', 'JPM', 'XOM', 'V', 'JNJ', 'MU','MA', 'COST', 'ORCL',
    'ABBV',  'BAC', 'HD', 'PG', 'CVX', 'GE', 'CAT',  'KO', 'NFLX', 'AMD','CSCO'
]
NUM_STOCKS = len(TICKERS)

MACRO_FEATURES            = ['VIX', 'TNX', 'SPY', 'QQQ', 'XLK']
INDICATORS_LIST           = INDICATORS
INDICATORS_WITH_SENT      = INDICATORS + ['sentiment']
INDICATORS_WITH_SENT_MACRO = INDICATORS + ['sentiment'] + MACRO_FEATURES

INDICATORS_COUNT_BASE  = len(INDICATORS_LIST)           # 8
INDICATORS_COUNT_SENT  = len(INDICATORS_WITH_SENT)      # 9
INDICATORS_COUNT_TFM   = len(INDICATORS_WITH_SENT_MACRO)# 14

# Alias used as default arg in class definitions
INDICATORS_COUNT       = INDICATORS_COUNT_SENT  # default = sentiment model (9)

INDICATORS_START_IDX = 1 + NUM_STOCKS + NUM_STOCKS
STATE_SPACE_BASE     = INDICATORS_START_IDX + (NUM_STOCKS * INDICATORS_COUNT_BASE)
STATE_SPACE_SENT     = INDICATORS_START_IDX + (NUM_STOCKS * INDICATORS_COUNT_SENT)

LOOKBACK_WINDOW = 10

print(f"NUM_STOCKS:       {NUM_STOCKS}")
print(f"STATE_SPACE base: {STATE_SPACE_BASE}")
print(f"STATE_SPACE sent: {STATE_SPACE_SENT}")

# ── VGG Feature Extractor ────────────────────────────────────
class VGG_FinRL_Extractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512, n_stocks=NUM_STOCKS, n_ind=INDICATORS_COUNT):
        super().__init__(observation_space, features_dim)
        self.n_stocks = n_stocks
        self.n_ind    = n_ind

        self.input_norm = nn.BatchNorm2d(1)

        self.vgg = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
        )

        # Probe passes through input_norm to stay consistent with forward()
        with th.no_grad():
            # Use CPU for the probe — MPS doesn't support BatchNorm in inference mode
            sample    = th.zeros(1, 1, n_stocks, n_ind)
            _norm_cpu = nn.BatchNorm2d(1)
            _vgg_cpu  = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
                nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            )
            sample    = _norm_cpu(sample)
            n_flatten = _vgg_cpu(sample).numel()

        # Dropout removed — it destabilises the RL critic's value estimates
        self.fc = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU()
        )

    def forward(self, observations):
        img_data = observations[:, INDICATORS_START_IDX:]
        img_data = img_data.view(-1, 1, self.n_stocks, self.n_ind)
        img_data = self.input_norm(img_data)
        x = self.vgg(img_data)
        x = th.flatten(x, start_dim=1)
        return self.fc(x)

# ── Transformer classes ───────────────────────────────────────
STATE_SPACE_TFM = INDICATORS_START_IDX + (NUM_STOCKS * INDICATORS_COUNT_TFM)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 100,
                 dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe       = th.zeros(max_len, d_model)
        position = th.arange(0, max_len).unsqueeze(1).float()
        div_term = th.exp(
            th.arange(0, d_model, 2).float() *
            (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = th.sin(position * div_term)
        pe[:, 1::2] = th.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ── Cross-Stock Transformer ───────────────────────────────────
class CrossStockTransformerFinRL(BaseFeaturesExtractor):
    """
    Two-stage transformer architecture:

    Stage 1 — Temporal attention (per stock):
      Each stock attends over its own history.
      With Polygon sentiment, learns patterns like:
        'NVDA has had 5 days of bullish news before a breakout'
        'AAPL sentiment turned negative 3 days before earnings miss'

    Stage 2 — Cross-stock attention (all stocks simultaneously):
      All stocks attend over each other.
      With real sentiment data, learns patterns like:
        'negative NVDA sentiment predicts AMD/AVGO moves'
        'high VIX + negative SPY predicts broad sell-off'

    This is the architecture that most benefits from full
    historical sentiment — it was previously operating blind.
    """
    def __init__(
        self,
        observation_space,
        features_dim:    int   = 128,
        n_stocks:        int   = NUM_STOCKS,
        n_indicators:    int   = INDICATORS_COUNT,
        d_model:         int   = 64,
        nhead:           int   = 4,
        num_layers:      int   = 2,
        dim_feedforward: int   = 256,
        dropout:         float = 0.1,
        lookback:        int   = LOOKBACK_WINDOW,
    ):
        super().__init__(observation_space, features_dim)
        self.lookback     = lookback
        self.n_stocks     = n_stocks
        self.n_indicators = n_indicators
        self.d_model      = d_model

        per_stock_dim = 2 + n_indicators

        self.stock_projection = nn.Sequential(
            nn.Linear(per_stock_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

        self.pos_encoder = PositionalEncoding(
            d_model, max_len=lookback + 1, dropout=dropout
        )

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model)
        )

        cross_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.cross_stock_transformer = nn.TransformerEncoder(
            cross_layer, num_layers=1,
            norm=nn.LayerNorm(d_model)
        )

        self.fc = nn.Sequential(
            nn.Linear(d_model * n_stocks, features_dim),
            nn.LayerNorm(features_dim),
            nn.ReLU()
        )

    def forward(self, observations: th.Tensor) -> th.Tensor:
        batch_size = observations.shape[0]
        single_dim = observations.shape[1] // self.lookback

        seq = observations.view(batch_size, self.lookback, single_dim)

        prices     = seq[:, :, 1:1+self.n_stocks]
        shares     = seq[:, :, 1+self.n_stocks:1+2*self.n_stocks]
        start_idx  = 1 + 2 * self.n_stocks
        indicators = seq[:, :, start_idx:].view(
            batch_size, self.lookback, self.n_stocks, self.n_indicators
        )

        prices_exp  = prices.unsqueeze(-1)
        shares_exp  = shares.unsqueeze(-1)
        stock_feats = th.cat([prices_exp, shares_exp, indicators], dim=-1)

        b, t, s, f  = stock_feats.shape
        stock_feats = self.stock_projection(
            stock_feats.view(b * t * s, f)
        ).view(b, t, s, self.d_model)

        # Stage 1: temporal attention
        stock_feats = stock_feats.permute(0, 2, 1, 3)
        stock_feats = stock_feats.reshape(b * s, t, self.d_model)
        stock_feats = self.pos_encoder(stock_feats)
        stock_feats = self.temporal_transformer(stock_feats)
        stock_feats = stock_feats[:, -1, :]
        stock_feats = stock_feats.view(b, s, self.d_model)

        # Stage 2: cross-stock attention
        stock_feats = self.cross_stock_transformer(stock_feats)

        return self.fc(stock_feats.reshape(b, s * self.d_model))


# ── Reward-shaped sequence environment ───────────────────────
class RewardShapedSequenceEnv(gymnasium.Wrapper):
    """
    Simplified reward shaping — four components only:
      1. 15% max concentration limit (action constraint)
      2. Sharpe reward — primary learning signal
      3. Single drawdown penalty at -10%
      4. Sentiment reward and penalty
    All other components removed to reduce reward complexity
    and allow the model to learn more clearly.
    """
    def __init__(self, env, df, lookback=LOOKBACK_WINDOW,
                 sharpe_weight=0.5,
                 drawdown_weight=0.3,
                 sentiment_weight=0.1):
        super().__init__(env)
        self.lookback         = lookback
        self.single_dim       = env.observation_space.shape[0]
        self._buffer          = np.zeros((lookback, self.single_dim),
                                         dtype=np.float32)
        self.sharpe_weight    = sharpe_weight
        self.drawdown_weight  = drawdown_weight
        self.sentiment_weight = sentiment_weight
        self._return_buffer   = []
        self._peak_value      = 100_000
        self._prev_value      = 100_000
        self.df               = df
        self.current_date_idx = 0
        self._sentiment_cache = {}

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(lookback * self.single_dim,),
            dtype=np.float32
        )

    def _get_stacked_obs(self):
        return self._buffer.flatten()

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs) \
                    if hasattr(self.env, 'reset') else (self.env.reset(), {})
        if isinstance(obs, tuple):
            obs = obs[0]
        self._buffer          = np.tile(obs, (self.lookback, 1))
        self._return_buffer   = []
        self._peak_value      = 100_000
        self._prev_value      = 100_000
        self.current_date_idx = 0

        # Pre-compute sentiment cache for entire episode
        try:
            self._sentiment_cache = {
                date: {
                    row['tic']: row['sentiment']
                    for _, row in day_df.iterrows()
                }
                for date, day_df in self.df.groupby('date')
            }
        except Exception:
            self._sentiment_cache = {}

        return self._get_stacked_obs(), {}

    def step(self, action):
        # ── Get today's sentiment from cache ─────────────────────
        today_sentiment = {}
        try:
            dates = self.df['date'].unique()
            if self.current_date_idx < len(dates):
                current_date    = dates[self.current_date_idx]
                today_sentiment = self._sentiment_cache.get(
                    current_date, {}
                )
        except (IndexError, KeyError):
            pass

        # ── Component 1: 15% max concentration limit ─────────────
        # Single action constraint — directly prevents the
        # over-concentration that caused the -93% drawdown.
        # Model learns freely within this structural guardrail.
        try:
            max_position_value = self._prev_value * 0.15
            for i in range(len(action)):
                try:
                    price = self.env.state[1 + i]
                    if price > 0:
                        max_shares = int(max_position_value / price)
                        action[i]  = int(np.clip(
                            action[i], -max_shares, max_shares
                        ))
                except (IndexError, AttributeError):
                    pass
        except Exception:
            pass

        result = self.env.step(action)
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result

        self._buffer = np.roll(self._buffer, shift=-1, axis=0)
        self._buffer[-1] = obs

        current_value = self.env.asset_memory[-1] \
                        if len(self.env.asset_memory) > 0 \
                        else self._prev_value

        daily_return = (current_value - self._prev_value) / \
                       (self._prev_value + 1e-8)
        self._return_buffer.append(daily_return)
        self._prev_value = current_value

        if len(self._return_buffer) > self.lookback:
            self._return_buffer = self._return_buffer[-self.lookback:]

        shaped_reward = reward

        # ── Component 2: Sharpe reward ────────────────────────────
        # Primary learning signal — teaches the model to maximise
        # risk-adjusted return. All profitable patterns learned
        # by the model flow from this signal.
        if len(self._return_buffer) >= 10:
            returns_arr  = np.array(self._return_buffer)
            ret_mean     = returns_arr.mean()
            ret_std      = returns_arr.std() + 1e-8
            sharpe_term  = ret_mean / ret_std * np.sqrt(252)
            shaped_reward += self.sharpe_weight * sharpe_term * 1e-3
            if sharpe_term < 0:
                shaped_reward += self.drawdown_weight * \
                                 sharpe_term * 1e-3

        # ── Component 3: Single drawdown penalty at -10% ─────────
        # Teaches the model to protect capital when gains erode.
        # Single clear threshold — no competing tiers.
        if current_value > self._peak_value:
            self._peak_value = current_value
        drawdown = (current_value - self._peak_value) / \
                   (self._peak_value + 1e-8)
        if drawdown < -0.10:
            shaped_reward += self.drawdown_weight * drawdown * 1e-1

        # ── Component 4: Sentiment reward and penalty ─────────────
        # Teaches the model to use Polygon news sentiment as a
        # trading signal — buy positive news, sell negative news,
        # penalise trading against sentiment.
        if today_sentiment:
            sentiment_reward  = 0.0
            sentiment_penalty = 0.0

            for i, ticker in enumerate(TICKERS):
                if ticker not in today_sentiment:
                    continue

                sent = today_sentiment[ticker]
                act  = action[i] if i < len(action) else 0

                # Reward: buying positive-news stocks
                if sent > 0.2 and act > 0:
                    sentiment_reward += sent * abs(act) * 1e-3

                # Reward: selling negative-news stocks
                if sent < -0.2 and act < 0:
                    sentiment_reward += abs(sent) * abs(act) * 1e-3

                # Penalty: buying negative-news stocks
                if sent < -0.2 and act > 0:
                    sentiment_penalty += abs(sent) * abs(act) * 2e-3

            shaped_reward += self.sentiment_weight * sentiment_reward
            shaped_reward -= self.sentiment_weight * sentiment_penalty

        self.current_date_idx += 1
        return self._get_stacked_obs(), shaped_reward, done, False, info


def make_env_sequence(df, lookback=LOOKBACK_WINDOW):
    base_env = StockTradingEnv(
        df=df,
        num_stock_shares=[0] * NUM_STOCKS,
        reward_scaling=1e-4,
        stock_dim=NUM_STOCKS,
        hmax=10,
        initial_amount=100_000,
        buy_cost_pct=[0.0015]  * NUM_STOCKS,   # 0.1% commission + 0.05% slippage
        sell_cost_pct=[0.0015] * NUM_STOCKS,
        state_space=STATE_SPACE_TFM,
        tech_indicator_list=INDICATORS_WITH_SENT_MACRO,
        action_space=NUM_STOCKS
    )
    return RewardShapedSequenceEnv(base_env, df=df, lookback=lookback)

# ── Environment factories ─────────────────────────────────────
def make_env_base(df):
    return StockTradingEnv(
        df=df,
        num_stock_shares    = [0] * NUM_STOCKS,
        reward_scaling      = 1e-4,
        stock_dim           = NUM_STOCKS,
        hmax                = 10,          # 100/$1M, 10/$100k, 5/$10k
        initial_amount      = 100_000,    # change per capital level
        buy_cost_pct        = [0.0015] * NUM_STOCKS,  # 0.1% + 0.05% slippage
        sell_cost_pct       = [0.0015] * NUM_STOCKS,
        state_space         = STATE_SPACE_BASE,
        tech_indicator_list = INDICATORS_LIST,
        action_space        = NUM_STOCKS
    )

def make_env_sent(df):
    return StockTradingEnv(
        df=df,
        num_stock_shares    = [0] * NUM_STOCKS,
        reward_scaling      = 1e-4,
        stock_dim           = NUM_STOCKS,
        hmax                = 10,          # 100/$1M, 10/$100k, 5/$10k
        initial_amount      = 100_000,    # change per capital level
        buy_cost_pct        = [0.0015] * NUM_STOCKS,  # 0.1% + 0.05% slippage
        sell_cost_pct       = [0.0015] * NUM_STOCKS,
        state_space         = STATE_SPACE_SENT,
        tech_indicator_list = INDICATORS_WITH_SENT,
        action_space        = NUM_STOCKS
    )

print("All architecture classes and environments defined ✅")


NUM_STOCKS:       30
STATE_SPACE base: 301
STATE_SPACE sent: 331
All architecture classes and environments defined ✅


In [11]:
# ============================================================
# CELL 3 — Build Test Environments
# Downloads / loads test data once — shared by all 12 models
# ============================================================

# ── Baseline + FinBERT test data (Yahoo Finance) ─────────────
print("Downloading Yahoo Finance test data...")
df_raw_test_yahoo = YahooDownloader(
    start_date='2024-01-01',
    end_date='2025-01-01',
    ticker_list=TICKERS
).fetch_data()

fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=False, use_turbulence=False
)
df_test_raw_yahoo = fe.preprocess_data(df_raw_test_yahoo)
df_test_base = prepare_df(df_test_raw_yahoo)
TICKERS_SORTED = sorted(df_test_base['tic'].unique().tolist())
NUM_STOCKS     = len(TICKERS_SORTED)

# Recalculate after locking tickers
INDICATORS_START_IDX = 1 + NUM_STOCKS + NUM_STOCKS
STATE_SPACE_BASE     = INDICATORS_START_IDX + (NUM_STOCKS * INDICATORS_COUNT_BASE)
STATE_SPACE_SENT     = INDICATORS_START_IDX + (NUM_STOCKS * INDICATORS_COUNT_SENT)
STATE_SPACE_TFM      = INDICATORS_START_IDX + (NUM_STOCKS * INDICATORS_COUNT_TFM)
print(f"Yahoo test env — {len(df_test_base['date'].unique())} trading days")

# ── VGG Alpaca test data (saved Alpaca CSV) ───────────────────
df_test_raw_alpaca = pd.read_csv('FinRLAlpaca_POLYGON__30Stocks_100k_validation.csv')
df_test_raw_alpaca = df_test_raw_alpaca[
    df_test_raw_alpaca['tic'].isin(TICKERS_SORTED)
].reset_index(drop=True)

if 'sentiment' in df_test_raw_alpaca.columns:
    df_test_sent = prepare_df(df_test_raw_alpaca, df_sentiment=None,
                              num_stocks=NUM_STOCKS)
    sent_vals = df_test_raw_alpaca.sort_values(['date', 'tic'])['sentiment'].values
    if len(sent_vals) == len(df_test_sent):
        df_test_sent['sentiment'] = sent_vals
    else:
        df_test_sent['sentiment'] = 0.0
    print("Sentiment extracted from Alpaca CSV ✅")
else:
    df_test_sent = prepare_df(df_test_raw_alpaca, df_sentiment=None,
                              num_stocks=NUM_STOCKS)
    df_test_sent['sentiment'] = 0.0
    print("No sentiment column — using zero fallback")

print(f"Alpaca sentiment test env — {len(df_test_sent['date'].unique())} trading days")

# ── Transformer test data (Alpaca + macro CSV) ────────────────
df_test_raw_tfm = pd.read_csv('FinRLTransformer_POLYGON_validation_30stocks_100k_seed3.csv')

# Do NOT filter by TICKERS_SORTED — Transformer uses a different 30-stock universe
TICKERS_TFM        = sorted(df_test_raw_tfm['tic'].unique().tolist())
NUM_STOCKS_TFM     = len(TICKERS_TFM)
INDICATORS_START_IDX_TFM = 1 + NUM_STOCKS_TFM + NUM_STOCKS_TFM
STATE_SPACE_TFM    = INDICATORS_START_IDX_TFM + (NUM_STOCKS_TFM * INDICATORS_COUNT_TFM)
print(f"Transformer universe: {NUM_STOCKS_TFM} tickers")

if 'sentiment' in df_test_raw_tfm.columns:
    df_test_tfm = prepare_df(df_test_raw_tfm, df_sentiment=None,
                             num_stocks=NUM_STOCKS_TFM)
    sent_vals = df_test_raw_tfm.sort_values(['date', 'tic'])['sentiment'].values
    if len(sent_vals) == len(df_test_tfm):
        df_test_tfm['sentiment'] = sent_vals
    else:
        df_test_tfm['sentiment'] = 0.0
else:
    df_test_tfm = prepare_df(df_test_raw_tfm, df_sentiment=None,
                             num_stocks=NUM_STOCKS_TFM)
    df_test_tfm['sentiment'] = 0.0

macro_cols = ['VIX', 'TNX', 'SPY', 'QQQ', 'XLK']
for col in macro_cols:
    if col in df_test_raw_tfm.columns:
        vals = df_test_raw_tfm.sort_values(['date', 'tic'])[col].values
        df_test_tfm[col] = vals if len(vals) == len(df_test_tfm) else 0.0
    else:
        df_test_tfm[col] = 0.0

n_macro = sum(1 for c in macro_cols if c in df_test_raw_tfm.columns)
print(f"Transformer test env — {len(df_test_tfm['date'].unique())} trading days  "
      f"({n_macro}/5 macro features from CSV)")

# ── Buy-and-hold baselines ────────────────────────────────────
df_bah_base = compute_buy_and_hold(df_test_base, INITIAL_CAPITAL)
df_bah_sent = compute_buy_and_hold(df_test_sent, INITIAL_CAPITAL)
df_bah_tfm  = compute_buy_and_hold(df_test_tfm,  INITIAL_CAPITAL)
print("\nAll test environments ready ✅")


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%*******

Shape of DataFrame:  (7560, 8)
Successfully added technical indicators
Using 30 tickers with complete data
No sentiment — baseline model
Index OK — starts at 0, ends at 251
Yahoo test env — 252 trading days
Using 30 tickers with complete data
No sentiment — baseline model
Index OK — starts at 0, ends at 251
Sentiment extracted from Alpaca CSV ✅
Alpaca sentiment test env — 252 trading days
Transformer universe: 30 tickers
Using 30 tickers with complete data
No sentiment — baseline model
Index OK — starts at 0, ends at 251
Transformer test env — 252 trading days  (5/5 macro features from CSV)
Buy-and-Hold: 30 stocks, $3,333 per stock
Buy-and-Hold: 30 stocks, $3,333 per stock
Buy-and-Hold: 30 stocks, $3,333 per stock

All test environments ready ✅


In [12]:
# ============================================================
# CELL 4 — Evaluation Helpers
# ============================================================

# ── Transformer predict function ─────────────────────────────
def predict_with_sequence_env(model, df, norm_path,
                               lookback=LOOKBACK_WINDOW):
    """
    Run prediction by applying VecNormalize stats manually.
    Bypasses the SB3 VecNormalize.load internal copy issue.
    """
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

    # Load normalization stats
    dummy_raw  = DummyVecEnv([lambda: make_env_sequence(df)])
    vec_norm   = VecNormalize.load(norm_path, dummy_raw)
    obs_rms    = vec_norm.obs_rms
    clip_obs   = vec_norm.clip_obs
    print(f"VecNormalize stats loaded ✅")
    print(f"  obs_rms.mean shape: {obs_rms.mean.shape}")

    # Build fresh direct environment
    env      = make_env_sequence(df)
    obs_raw  = env.reset()
    obs      = obs_raw[0] if isinstance(obs_raw, tuple) else obs_raw

    raw_actions = []
    done        = False
    n_steps     = 0
    max_steps   = len(df['date'].unique()) + 10

    while not done and n_steps < max_steps:
        # Apply normalization manually
        obs_normalized = np.clip(
            (obs - obs_rms.mean) / np.sqrt(obs_rms.var + 1e-8),
            -clip_obs, clip_obs
        ).astype(np.float32)

        action, _ = model.predict(
            obs_normalized.reshape(1, -1), deterministic=True
        )
        raw_actions.append(action[0].copy())

        result = env.step(action[0])
        if len(result) == 5:
            obs, _, terminated, truncated, _ = result
            done = terminated or truncated
        else:
            obs, _, done, _ = result

        if isinstance(obs, tuple):
            obs = obs[0]

        n_steps += 1

    print(f"Episode finished at step {n_steps}")

    # Get base env memory
    base_env = env
    while hasattr(base_env, 'env'):
        base_env = base_env.env

    print(f"asset_memory length:   {len(base_env.asset_memory)}")

    # Build output DataFrames
    dates        = sorted(df['date'].unique())
    asset_memory = base_env.asset_memory

    account_values = asset_memory[1:] \
                     if len(asset_memory) > len(dates) \
                     else asset_memory

    min_len = min(len(dates), len(account_values))

    df_account_value = pd.DataFrame({
        'date':          dates[:min_len],
        'account_value': account_values[:min_len]
    })

    action_min_len = min(min_len, len(raw_actions))
    if raw_actions and len(raw_actions[0]) > 0:
        n_act     = len(raw_actions[0])
        col_names = TICKERS[:n_act] if n_act == len(TICKERS) \
                    else [f'action_{i}' for i in range(n_act)]
        df_actions = pd.DataFrame(
            raw_actions[:action_min_len],
            columns=col_names
        )
    else:
        df_actions = pd.DataFrame()

    if len(df_account_value) > 1:
        start_val = df_account_value['account_value'].iloc[0]
        end_val   = df_account_value['account_value'].iloc[-1]
        total_ret = (end_val - start_val) / start_val * 100
        print(f"Prediction complete — {len(df_account_value)} days")
        print(f"Start: ${start_val:,.0f}  "
              f"End:   ${end_val:,.0f}  "
              f"Return: {total_ret:.2f}%")

    return df_account_value, df_actions

def evaluate_and_save(model, test_env, df_test_raw,
                       df_bah, model_label, seed,
                       has_sentiment, is_transformer=False,
                       norm_path=None):
    """
    Full evaluation for one model/seed combination.
    Produces all metrics identical to run_full_evaluation().
    Returns dict of results.
    """
    save_key = f"{model_label}_seed{seed}"
    print(f"\n============================================================")
    print(f"  EVALUATING: {save_key}")
    print(f"============================================================")

    # ── Run predictions ───────────────────────────────────────
    if is_transformer:
        df_test_account, df_actions = predict_with_sequence_env(
            model, df_test_raw, norm_path=norm_path
        )
        # Transformer has no training env in collector — skip train metrics
        df_train_account = None
        train_metrics    = None
    else:
        df_test_account, df_actions = DRLAgent.DRL_prediction(
            model=model, environment=test_env
        )
        df_train_account = None
        train_metrics    = None

    # ── Truncate to peak ─────────────────────────────────────
    peak_idx     = df_test_account['account_value'].idxmax()
    peak_date    = df_test_account.loc[peak_idx, 'date']
    df_test_peak = df_test_account.loc[:peak_idx].reset_index(drop=True)
    df_act_peak  = df_actions.iloc[:peak_idx+1].reset_index(drop=True)
    print(f"  Peak date: {peak_date}")

    # ── Compute metrics ───────────────────────────────────────
    test_metrics = compute_metrics(df_test_peak,    INITIAL_CAPITAL)
    full_metrics = compute_metrics(df_test_account, INITIAL_CAPITAL)
    bah_metrics  = compute_metrics(df_bah,          INITIAL_CAPITAL)

    print(f"\n  TEST METRICS (to peak)")
    for k, v in test_metrics.items():
        print(f"    {k:<25} {v:>10}")

    print(f"\n  FULL PERIOD METRICS")
    for k, v in full_metrics.items():
        print(f"    {k:<25} {v:>10}")

    print(f"\n  BUY-AND-HOLD")
    for k, v in bah_metrics.items():
        print(f"    {k:<25} {v:>10}")

    vs_bah = test_metrics['Sharpe Ratio'] - bah_metrics['Sharpe Ratio']
    print(f"\n  vs Buy-and-Hold Sharpe: {'+' if vs_bah>=0 else ''}{vs_bah:.3f}")

    # ── Degenerate policy check ───────────────────────────────
    check_degenerate_policy(df_act_peak, df_test_peak, model_name=save_key)

    # ── Regime analysis ───────────────────────────────────────
    regime_analysis(df_test_peak, df_bah,
                    initial_capital=INITIAL_CAPITAL,
                    model_name=save_key)

    # ── Rolling metrics + plots ───────────────────────────────
    test_rolling = compute_rolling_metrics(df_test_peak, INITIAL_CAPITAL)
    plot_metrics(test_rolling,
                 title_prefix=f"{save_key} — Test (to peak)")

    # ── Save pkl files ────────────────────────────────────────
    for fname, obj in {
        f'{save_key}_test_rolling':  test_rolling,
        f'{save_key}_account_value': df_test_peak,
        f'{save_key}_bah':           df_bah,
        f'{save_key}_actions':       df_act_peak,
    }.items():
        with open(f'./overlay_data/{fname}.pkl', 'wb') as fh:
            pickle.dump(obj, fh)

    # ── Save metrics summary text file ────────────────────────
    summary_path = f'./overlay_data/{save_key}_metrics_summary.txt'
    with open(summary_path, 'w') as fh:
        fh.write(f"MODEL:         {save_key}\n")
        fh.write(f"Has sentiment: {has_sentiment}\n")
        fh.write(f"Peak date:     {peak_date}\n")
        fh.write(f"Capital:       ${INITIAL_CAPITAL:,.0f}\n\n")
        fh.write("TEST METRICS (to peak)\n" + "-"*35 + "\n")
        for k, v in test_metrics.items():
            fh.write(f"  {k}: {v}\n")
        fh.write("\nFULL PERIOD METRICS\n" + "-"*35 + "\n")
        for k, v in full_metrics.items():
            fh.write(f"  {k}: {v}\n")
        fh.write("\nBUY-AND-HOLD METRICS\n" + "-"*35 + "\n")
        for k, v in bah_metrics.items():
            fh.write(f"  {k}: {v}\n")

    print(f"  Saved: {summary_path}")
    print(f"============================================================\n")

    return {
        'save_key':       save_key,
        'test_metrics':   test_metrics,
        'full_metrics':   full_metrics,
        'bah_metrics':    bah_metrics,
        'test_rolling':   test_rolling,
        'df_test_peak':   df_test_peak,
        'df_bah':         df_bah,
        'df_actions':     df_act_peak,
        'peak_date':      peak_date,
    }

print("Evaluation helpers defined ✅")


Evaluation helpers defined ✅


In [15]:
# ============================================================
# CELL 5 — Run All 12 Evaluations
# Run only after all 12 seed notebooks have finished training
# ============================================================

all_results = {}   # {model_label: [result_seed1, result_seed2, result_seed3]}

# ── VGG Baseline ─────────────────────────────────────────────
print("\n" + "="*60)
print("VGG BASELINE — 3 Seeds")
print("="*60)
baseline_results = []
for seed in SEEDS:
    model = PPO.load(f'./best_vgg_baseline_30stocks_100k_seed{seed}',
                     device=DEVICE)
    env   = make_env_base(df_test_base)
    res   = evaluate_and_save(
        model=model, test_env=env,
        df_test_raw=df_test_base, df_bah=df_bah_base,
        model_label='30_Stock_100k_VGG_Baseline',
        seed=seed, has_sentiment=False
    )
    baseline_results.append(res)
all_results['VGG Baseline'] = baseline_results

# ── VGG FinBERT ───────────────────────────────────────────────
print("\n" + "="*60)
print("VGG FINBERT — 3 Seeds")
print("="*60)
finbert_results = []
for seed in SEEDS:
    model = PPO.load(f'./best_vgg_finbert_30stocks_100k_seed{seed}',
                     device=DEVICE)
    env   = make_env_sent(df_test_sent)
    res   = evaluate_and_save(
        model=model, test_env=env,
        df_test_raw=df_test_sent, df_bah=df_bah_sent,
        model_label='30_Stock_100k_VGG_FinBERT',
        seed=seed, has_sentiment=True
    )
    finbert_results.append(res)
all_results['VGG FinBERT'] = finbert_results

# ── VGG Alpaca ────────────────────────────────────────────────
print("\n" + "="*60)
print("VGG ALPACA — 3 Seeds")
print("="*60)
alpaca_results = []
for seed in SEEDS:
    model = PPO.load(f'./best_vgg_alpaca_30stocks_100k_seed{seed}',
                     device=DEVICE)
    env   = make_env_sent(df_test_sent)
    res   = evaluate_and_save(
        model=model, test_env=env,
        df_test_raw=df_test_sent, df_bah=df_bah_sent,
        model_label='30_Stock_100k_VGG_Alpaca',
        seed=seed, has_sentiment=True
    )
    alpaca_results.append(res)
all_results['VGG Alpaca'] = alpaca_results

# ── Transformer ───────────────────────────────────────────────
print("\n" + "="*60)
print("TRANSFORMER — 3 Seeds")
print("="*60)
tfm_results = []
for seed in SEEDS:
    model_path = f'./best_train_sharpe_30stocks_100k_seed{seed}/best_train_model'
    norm_path  = f'vec_normalize_30stocks_100k_seed{seed}.pkl'
    model      = PPO.load(model_path, device='cpu')
    model.policy.to('cpu')
    env = make_env_sequence(df_test_tfm)
    res = evaluate_and_save(
        model=model, test_env=env,
        df_test_raw=df_test_tfm, df_bah=df_bah_sent,
        model_label='30_Stock_100k_Transformer',
        seed=seed, has_sentiment=True,
        is_transformer=True, norm_path=norm_path
    )
    tfm_results.append(res)
all_results['Transformer'] = tfm_results

print("\n✅ All 12 evaluations complete")



VGG BASELINE — 3 Seeds


/opt/miniconda3/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(



  EVALUATING: 30_Stock_100k_VGG_Baseline_seed1
hit end!
  Peak date: 2024-12-24

  TEST METRICS (to peak)
    Total Return (%)               45.53
    Sharpe Ratio                   2.045
    Max Drawdown (%)              -10.29
    Win Rate (%)                   62.35
    Avg Daily Return (%)          0.1578
    Volatility (%)                 16.99
    Calmar Ratio                   4.425

  FULL PERIOD METRICS
    Total Return (%)               42.37
    Sharpe Ratio                   1.885
    Max Drawdown (%)              -10.29
    Win Rate (%)                   61.35
    Avg Daily Return (%)          0.1465
    Volatility (%)                 16.93
    Calmar Ratio                   4.117

  BUY-AND-HOLD
    Total Return (%)               39.34
    Sharpe Ratio                   1.974
    Max Drawdown (%)              -10.82
    Win Rate (%)                   57.37
    Avg Daily Return (%)          0.1367
    Volatility (%)                 14.91
    Calmar Ratio                  

/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:583: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:792: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:360: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



  30_STOCK_100K_VGG_BASELINE_SEED1 — TEST (TO PEAK) — FINAL METRIC VALUES
  Sharpe Ratio                   2.299  (avg)
  Win Rate (%)                  61.908  (avg)
  Max Drawdown (%)              -3.143  (avg)
  Avg Daily Return (%)           0.147  (avg)
  Total Return (%)              45.533  (final)
  Saved: ./overlay_data/30_Stock_100k_VGG_Baseline_seed1_metrics_summary.txt



...

/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:583: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:792: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:360: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



  30_STOCK_100K_TRANSFORMER_SEED2 — TEST (TO PEAK) — FINAL METRIC VALUES
  Sharpe Ratio                   1.424  (avg)
  Win Rate (%)                  59.086  (avg)
  Max Drawdown (%)              -3.171  (avg)
  Avg Daily Return (%)           0.098  (avg)
  Total Return (%)              28.720  (final)
  Saved: ./overlay_data/30_Stock_100k_Transformer_seed2_metrics_summary.txt


  EVALUATING: 30_Stock_100k_Transformer_seed3
VecNormalize stats loaded ✅
  obs_rms.mean shape: (4810,)
Episode finished at step 252
asset_memory length:   252
Prediction complete — 252 days
Start: $100,000  End:   $116,306  Return: 16.31%
  Peak date: 2024-11-08

  TEST METRICS (to peak)
    Total Return (%)               30.09
    Sharpe Ratio                   1.815
    Max Drawdown (%)               -7.44
    Win Rate (%)                   53.24
    Avg Daily Return (%)          0.1262
    Volatility (%)                 14.76
    Calmar Ratio                   4.046

  FULL PERIOD METRICS
    Total Return

/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:583: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:792: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



  30_STOCK_100K_TRANSFORMER_SEED3 — TEST (TO PEAK) — FINAL METRIC VALUES
  Sharpe Ratio                   1.816  (avg)
  Win Rate (%)                  52.741  (avg)
  Max Drawdown (%)              -3.181  (avg)
  Avg Daily Return (%)           0.105  (avg)
  Total Return (%)              30.094  (final)
  Saved: ./overlay_data/30_Stock_100k_Transformer_seed3_metrics_summary.txt


✅ All 12 evaluations complete


/Users/tylerhobbs/Documents/Virginia/DS6050/Project/Code/utils.py:360: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# ============================================================
# CELL 6 — Multi-Seed Summary Table
# ============================================================

# All metric keys from compute_metrics
METRIC_KEYS = [
    'Sharpe Ratio', 'Total Return (%)', 'Max Drawdown (%)',
    'Win Rate (%)', 'Volatility (%)', 'Calmar Ratio',
    'Avg Daily Return (%)'
]

rows = []
for model_name, results in all_results.items():
    for metric in METRIC_KEYS:
        vals = [r['test_metrics'][metric] for r in results]
        rows.append({
            'Model':   model_name,
            'Metric':  metric,
            'Seed 1':  vals[0],
            'Seed 2':  vals[1],
            'Seed 3':  vals[2],
            'Mean':    round(np.mean(vals), 3),
            'Std':     round(np.std(vals), 3),
        })

df_summary = pd.DataFrame(rows)
df_summary.to_csv('multi_seed_full_metrics_30stocks_100k.csv', index=False)
print("Full metrics saved: multi_seed_full_metrics_30stocks_100k.csv")

# ── Print formatted table for the paper ──────────────────────
print()
print("=" * 80)
print("MULTI-SEED RESULTS — Test Period 2024 (to peak) — 30-Stock $100k")
print("=" * 80)

for model_name, results in all_results.items():
    print(f"\n  {model_name}")
    print(f"  {'Metric':<25} {'S1':>8} {'S2':>8} {'S3':>8} {'Mean':>8} {'Std':>8} {'Reliable':>12}")
    print(f"  {'-'*77}")
    for metric in METRIC_KEYS:
        vals  = [r['test_metrics'][metric] for r in results]
        mean  = np.mean(vals)
        std   = np.std(vals)
        flag  = "✅" if std < 0.3 else ("⚠" if std < 0.5 else "❌")
        print(f"  {metric:<25} {vals[0]:>8.3f} {vals[1]:>8.3f} {vals[2]:>8.3f} "
              f"{mean:>8.3f} {std:>8.3f} {flag:>12}")

print("=" * 80)

# ── Single-seed vs multi-seed Sharpe comparison ──────────────
single_seed = {
    'VGG Baseline': 2.287,
    'VGG FinBERT':  2.350,
    'VGG Alpaca':   2.531,
    'Transformer':  1.468,
}
print()
print("TABLE FOR PAPER — Sharpe Ratio comparison")
print("=" * 60)
print(f"{'Model':<20} {'Original':>10} {'Mean±Std':>15} {'Verdict':>15}")
print("-" * 60)
for model_name, results in all_results.items():
    vals = [r['test_metrics']['Sharpe Ratio'] for r in results]
    mean = np.mean(vals)
    std  = np.std(vals)
    orig = single_seed.get(model_name, 0)
    verdict = "Confirms" if abs(mean - orig) < std * 2 else "Diverges"
    print(f"{model_name:<20} {orig:>10.3f} {mean:>8.3f}±{std:<6.3f} {verdict:>15}")
print("=" * 60)


Full metrics saved: multi_seed_full_metrics_30stocks_100k.csv

MULTI-SEED RESULTS — Test Period 2024 (to peak) — 30-Stock $100k

  VGG Baseline
  Metric                          S1       S2       S3     Mean      Std     Reliable
  -----------------------------------------------------------------------------
  Sharpe Ratio                 2.045    2.166    2.196    2.136    0.065            ✅
  Total Return (%)            45.530   47.950   42.240   45.240    2.340            ❌
  Max Drawdown (%)           -10.290   -9.920   -9.630   -9.947    0.270            ✅
  Win Rate (%)                62.350   59.660   56.300   59.437    2.475            ❌
  Volatility (%)              16.990   17.560   15.260   16.603    0.978            ❌
  Calmar Ratio                 4.425    4.833    4.387    4.548    0.202            ✅
  Avg Daily Return (%)         0.158    0.171    0.153    0.160    0.008            ✅

  VGG FinBERT
  Metric                          S1       S2       S3     Mean      Std 

In [17]:
# ============================================================
# CELL 7 — Visualisation
# ============================================================
models  = list(all_results.keys())
colors  = ['#00d68f', '#63b3ed', '#f6ad55', '#ff6b9d']

# ── Plot 1: Sharpe mean±std ───────────────────────────────────
sharpes = {m: [r['test_metrics']['Sharpe Ratio']
               for r in all_results[m]] for m in models}
means   = [np.mean(sharpes[m]) for m in models]
stds    = [np.std(sharpes[m])  for m in models]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Multi-Seed Evaluation — 30-Stock $100k\nTest Period 2024",
             fontsize=13, fontweight='bold')

ax1 = axes[0]
x   = np.arange(len(models))
bars = ax1.bar(x, means, yerr=stds, capsize=8, color=colors,
               alpha=0.85, width=0.55,
               error_kw={'linewidth': 2, 'ecolor': 'black'})
ax1.axhline(1.975, color='gray', linestyle='--', linewidth=1.5,
            label='Buy-and-Hold (1.975)')
ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=10)
ax1.set_ylabel('Test Sharpe Ratio', fontsize=11)
ax1.set_title('Mean ± Std Sharpe (3 seeds)', fontsize=11)
ax1.legend(fontsize=9)
ax1.set_ylim(0, max(means) + max(stds) + 0.6)
ax1.grid(axis='y', alpha=0.3)
for bar, mean in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.05,
             f'{mean:.3f}', ha='center', va='bottom',
             fontsize=10, fontweight='bold')

ax2 = axes[1]
for idx, (model, color) in enumerate(zip(models, colors)):
    for si, s in enumerate(sharpes[model]):
        x_pos = idx + (si - 1) * 0.12
        ax2.scatter(x_pos, s, color=color, s=90, zorder=3)
    mean_s = np.mean(sharpes[model])
    ax2.hlines(mean_s, idx - 0.2, idx + 0.2,
               colors=color, linewidth=2.5, zorder=4)
ax2.axhline(1.975, color='gray', linestyle='--', linewidth=1.5,
            label='Buy-and-Hold')
ax2.set_xticks(range(len(models)))
ax2.set_xticklabels(models, fontsize=10)
ax2.set_ylabel('Test Sharpe Ratio', fontsize=11)
ax2.set_title('Individual Seeds (dots) + Mean (line)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('multi_seed_sharpe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: multi_seed_sharpe_comparison.png")

# ── Plot 2: All metrics heatmap ───────────────────────────────
metric_means = {}
for model_name, results in all_results.items():
    metric_means[model_name] = {
        metric: np.mean([r['test_metrics'][metric] for r in results])
        for metric in METRIC_KEYS
    }

df_heat = pd.DataFrame(metric_means).T
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("Mean Metrics Across 3 Seeds — 30-Stock $100k",
             fontsize=13, fontweight='bold')

normalized = df_heat.copy()
for col in normalized.columns:
    col_min = normalized[col].min()
    col_max = normalized[col].max()
    if col_max != col_min:
        normalized[col] = (normalized[col] - col_min) / (col_max - col_min)
    else:
        normalized[col] = 0.5

im = ax.imshow(normalized.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(METRIC_KEYS)))
ax.set_xticklabels(METRIC_KEYS, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models, fontsize=10)
for i, model in enumerate(models):
    for j, metric in enumerate(METRIC_KEYS):
        val = df_heat.loc[model, metric]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, fontweight='bold')
plt.colorbar(im, ax=ax, label='Normalised score (green=best)')
plt.tight_layout()
plt.savefig('multi_seed_metrics_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: multi_seed_metrics_heatmap.png")


Figure saved: multi_seed_sharpe_comparison.png
Figure saved: multi_seed_metrics_heatmap.png


/var/folders/sc/40gcq14x2cng8czg1s_vpkbr0000gn/T/ipykernel_10545/1957576940.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/sc/40gcq14x2cng8czg1s_vpkbr0000gn/T/ipykernel_10545/1957576940.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
